# LeetCode 3504 — Longest Palindrome After Substring Concatenation II bằng Python

- Link: https://leetcode.com/problems/longest-palindrome-after-substring-concatenation-ii/
- Độ khó: **Hard**
- Chủ đề: **Palindrome** (mở rộng từ tâm), **DP hai chuỗi** kiểu "khớp đối xứng" `s[i]` với `t[j]`, palindrome dài nhất **bắt đầu tại** / **kết thúc tại** một vị trí

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def longestPalindrome(self, s: str, t: str) -> int:
```

## 1. Mục tiêu học

1. Phân tích cấu trúc của palindrome `A + B` (với `A` là chuỗi con của `s`, `B` là chuỗi con của `t`): phần ngoài "khớp gương" giữa `A` và `B`, phần giữa là một palindrome nằm **hoàn toàn** trong `A` hoặc trong `B`.
2. Tính `p[i]` = palindrome dài nhất của `s` **bắt đầu** tại `i`, `q[j]` = palindrome dài nhất của `t` **kết thúc** tại `j` (mở rộng từ tâm `O(n^2)`).
3. Viết DP `dp[i][j]` = palindrome dài nhất có ký tự đầu là `s[i]` (hoặc rỗng phía `s`) và ký tự cuối là `t[j]`, theo công thức của gợi ý đề.
4. Cài DP `O(n·m)` với hai hàng (tiết kiệm bộ nhớ).
5. Kiểm chứng bằng vét cạn mọi cặp chuỗi con.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s`, `t`. Chọn một chuỗi con (liên tiếp, có thể rỗng) của `s` và một chuỗi con (có thể rỗng) của `t`, nối **theo thứ tự** `s` trước `t`. Trả về độ dài palindrome dài nhất tạo được.

**Ví dụ 1:** `s = "a", t = "a"` → `2`.

**Ví dụ 2:** `s = "abc", t = "def"` → `1`.

**Ví dụ 3:** `s = "b", t = "aaaa"` → `4`.

**Ví dụ 4:** `s = "abcde", t = "ecdba"` → `5` (`"abc" + "ba" = "abcba"`).

**Ràng buộc:** `1 ≤ |s|, |t| ≤ 1000`, chữ thường.

In [1]:
examples = [("a", "a", 2), ("abc", "def", 1), ("b", "aaaa", 4), ("abcde", "ecdba", 5)]
for s, t, exp in examples:
    print(repr(s), repr(t), '-> kỳ vọng', exp)

'a' 'a' -> kỳ vọng 2
'abc' 'def' -> kỳ vọng 1
'b' 'aaaa' -> kỳ vọng 4
'abcde' 'ecdba' -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Mở rộng từ tâm

Mỗi palindrome có tâm (một ký tự hoặc giữa hai ký tự). Duyệt `2n − 1` tâm, mở rộng khi hai đầu bằng nhau ⇒ liệt kê mọi palindrome trong `O(n^2)`. Ghi lại cho mỗi vị trí bắt đầu `l` độ dài lớn nhất.

In [2]:
def longest_pal_starting(s):
    n = len(s)
    p = [1] * n
    for c in range(2 * n - 1):
        l, r = c // 2, c // 2 + (c & 1)
        while l >= 0 and r < n and s[l] == s[r]:
            p[l] = max(p[l], r - l + 1)
            l -= 1; r += 1
    return p

def longest_pal_ending(t):
    return longest_pal_starting(t[::-1])[::-1]

print(longest_pal_starting("abaab"), longest_pal_ending("abaab"))

[3, 4, 2, 1, 1] [1, 1, 3, 2, 4]


### 3.2. Cấu trúc của palindrome `A + B`

Giả sử `|A| ≥ |B|`. Vì `A + B` là palindrome: `B` đọc ngược bằng `|B|` ký tự **đầu** của `A`, và phần còn lại của `A` (ở giữa) là palindrome. Tương tự nếu `|B| > |A|` thì phần giữa nằm trong `B`.

In [3]:
A, B = "abc", "ba"
print(A + B, '| B ngược =', B[::-1], '= đầu của A:', A[:len(B)], '| phần giữa:', A[len(B):])

abcba | B ngược = ab = đầu của A: ab | phần giữa: c


### 3.3. DP trên cặp chỉ số đi ngược chiều

Ghép `s[i]` (đi sang phải) với `t[j]` (đi sang trái): nếu bằng nhau thì có thể "bọc" thêm hai ký tự vào palindrome tốt nhất của cặp `(i+1, j−1)`.

### 3.4. Hai hàng DP

`dp[i][·]` chỉ phụ thuộc `dp[i+1][·]` ⇒ giữ hai mảng độ dài `m + 1`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 4: `s = "abcde"`, `t = "ecdba"`. Ghép gương từ ngoài vào: `s[0] = 'a'` với `t[4] = 'a'`, `s[1] = 'b'` với `t[3] = 'b'`. Sau đó dừng, phần giữa là palindrome bắt đầu tại `s[2]` (`"c"`) hoặc kết thúc tại `t[2]` (`"d"`) ⇒ độ dài `2·2 + 1 = 5`.

Định nghĩa (theo gợi ý đề) `dp[i][j]` = độ dài lớn nhất của palindrome `s[i..] + t[..j]` (bắt đầu đúng tại `s[i]` nếu dùng `s`, kết thúc đúng tại `t[j]` nếu dùng `t`):

- luôn có thể chỉ dùng palindrome trong `s` bắt đầu tại `i`: `p[i]`, hoặc trong `t` kết thúc tại `j`: `q[j]`;
- nếu `s[i] = t[j]`: bọc thêm `2 + dp[i+1][j−1]`.

In [4]:
s, t = "abcde", "ecdba"
p, q = longest_pal_starting(s), longest_pal_ending(t)
print('p =', p, '| q =', q)
print("s[0] = t[4]:", s[0] == t[4], "| s[1] = t[3]:", s[1] == t[3], "| s[2] = t[2]:", s[2] == t[2])
print('2·2 + max(p[2], q[2]) =', 4 + max(p[2], q[2]))

p = [1, 1, 1, 1, 1] | q = [1, 1, 1, 1, 1]
s[0] = t[4]: True | s[1] = t[3]: True | s[2] = t[2]: False
2·2 + max(p[2], q[2]) = 5


## 5. Cách 1 — Brute Force: thử mọi cặp chuỗi con

In [5]:
def longest_brute(s, t):
    best = 0
    for a in range(len(s) + 1):
        for b in range(a, len(s) + 1):
            for c in range(len(t) + 1):
                for d in range(c, len(t) + 1):
                    w = s[a:b] + t[c:d]
                    if len(w) > best and w == w[::-1]:
                        best = len(w)
    return best

for s, t, exp in examples:
    print(longest_brute(s, t), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
4 | kỳ vọng 4
5 | kỳ vọng 5


**Độ phức tạp:** `O(n^2 m^2 (n + m))` — chỉ dùng với chuỗi rất ngắn.

**Nút thắt:** phần "giữa" và phần "gương" bị kiểm tra lại từ đầu cho mỗi cặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề cấu trúc (mục 3.2):** mọi palindrome `A + B` có dạng `X + M + reverse(X)` trong đó `X` là tiền tố của `A`, `reverse(X)` là hậu tố của `B`, và `M` là palindrome nằm trọn trong phần còn lại của `A` **hoặc** của `B`.

⇒ Với điểm bắt đầu `i` (trong `s`) và kết thúc `j` (trong `t`):

`dp[i][j] = max(p[i], q[j], 2 + dp[i+1][j−1] nếu s[i] = t[j])`,

biên: `dp[n][j] = q[j]` (hết `s`), `dp[i][−1] = p[i]` (hết `t`), quy ước `p[n] = q[−1] = 0`.

Đáp án `= max` mọi `dp[i][j]` (cũng bao gồm `max(p)`, `max(q)`).

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi cặp `(i, j)` và mở rộng gương thủ công (`O(n·m·min(n, m))`)

Với mỗi cặp bắt đầu `(i, j)`, đi vào trong khi `s[i+l] = t[j−l]`; tại mỗi độ dài gương `l` (kể cả `0`), ứng viên `2l + max(p[i+l], q[j−l])`.

In [6]:
def longest_extend(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_ending(t)              # q[j + 1] ứng với t[j]; q[0] = 0
    best = max(max(p), max(q))
    for i in range(n + 1):
        for j in range(-1, m):
            l = 0
            while True:
                best = max(best, 2 * l + max(p[i + l], q[j - l + 1]))
                if i + l < n and j - l >= 0 and s[i + l] == t[j - l]:
                    l += 1
                else:
                    break
    return best

for s, t, exp in examples:
    print(longest_extend(s, t), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
4 | kỳ vọng 4
5 | kỳ vọng 5


**Độ phức tạp:** `O(n · m · L)` với `L` là độ dài gương lớn nhất — xấu nhất `10^9` khi hai chuỗi toàn `'a'`.

### 7.2. Cách 3 — DP `dp[i][j]` đầy đủ (lời giải chính, dạng bảng)

In [7]:
def longest_dp_table(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_ending(t)              # chỉ số dịch 1: jj = j + 1
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for jj in range(m + 1):
        dp[n][jj] = q[jj]
    best = max(q)
    for i in range(n - 1, -1, -1):
        dp[i][0] = p[i]
        for jj in range(1, m + 1):
            v = max(p[i], q[jj])
            if s[i] == t[jj - 1]:
                v = max(v, 2 + dp[i + 1][jj - 1])
            dp[i][jj] = v
        best = max(best, max(dp[i]))
    return best

for s, t, exp in examples:
    print(longest_dp_table(s, t), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
4 | kỳ vọng 4
5 | kỳ vọng 5


**Độ phức tạp:** `O(n·m)` thời gian (cộng `O(n^2 + m^2)` cho `p`, `q`), `O(n·m)` bộ nhớ.

### 7.3. Cách 4 — DP hai hàng (tiết kiệm bộ nhớ)

In [8]:
def longest_dp_rolling(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_ending(t)
    nxt = q[:]                                   # dp[n][*]
    best = max(max(p), max(q))
    for i in range(n - 1, -1, -1):
        cur = [p[i]] + [0] * m
        si, pi = s[i], p[i]
        for jj in range(1, m + 1):
            v = pi if pi > q[jj] else q[jj]
            if si == t[jj - 1]:
                w = 2 + nxt[jj - 1]
                if w > v:
                    v = w
            cur[jj] = v
        mx = max(cur)
        if mx > best:
            best = mx
        nxt = cur
    return best

for s, t, exp in examples:
    print(longest_dp_rolling(s, t), '| kỳ vọng', exp)

2 | kỳ vọng 2
1 | kỳ vọng 1
4 | kỳ vọng 4
5 | kỳ vọng 5


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 4: in `p`, `q` và bảng `dp` (hàng `i` từ dưới lên, cột `j` của `t`).

In [9]:
def trace(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_ending(t)
    print('p (bắt đầu trong s) =', p[:-1], '| q (kết thúc trong t) =', q[1:])
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    dp[n] = q[:]
    for i in range(n - 1, -1, -1):
        dp[i][0] = p[i]
        for jj in range(1, m + 1):
            v = max(p[i], q[jj])
            if s[i] == t[jj - 1]:
                v = max(v, 2 + dp[i + 1][jj - 1])
            dp[i][jj] = v
    print('      ' + '  '.join(f"t[{j}]={t[j]}" for j in range(m)))
    for i in range(n):
        print(f"s[{i}]={s[i]}  " + '      '.join(str(dp[i][jj]) for jj in range(1, m + 1)))
    print('đáp án =', max(max(r) for r in dp))

trace("abcde", "ecdba")

p (bắt đầu trong s) = [1, 1, 1, 1, 1] | q (kết thúc trong t) = [1, 1, 1, 1, 1]
      t[0]=e  t[1]=c  t[2]=d  t[3]=b  t[4]=a
s[0]=a  1      1      1      1      5
s[1]=b  1      1      1      3      1
s[2]=c  1      3      1      1      1
s[3]=d  1      1      3      1      1
s[4]=e  2      1      1      1      1
đáp án = 5


Ô `(s[0]='a', t[4]='a')` = `2 + dp[1][3]` = `2 + (2 + max(p[2], q[2]))` = `5`.

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def longestPalindrome(self, s: str, t: str) -> int:
        def pal_start(x):
            n = len(x)
            res = [1] * n
            for c in range(2 * n - 1):
                l, r = c // 2, c // 2 + (c & 1)
                while l >= 0 and r < n and x[l] == x[r]:
                    if r - l + 1 > res[l]:
                        res[l] = r - l + 1
                    l -= 1
                    r += 1
            return res

        n, m = len(s), len(t)
        p = pal_start(s) + [0]
        q = [0] + pal_start(t[::-1])[::-1]
        nxt = q[:]
        best = max(max(p), max(q))
        for i in range(n - 1, -1, -1):
            pi, si = p[i], s[i]
            cur = [pi] + [0] * m
            for jj in range(1, m + 1):
                v = pi if pi > q[jj] else q[jj]
                if si == t[jj - 1]:
                    w = 2 + nxt[jj - 1]
                    if w > v:
                        v = w
                cur[jj] = v
            mx = max(cur)
            if mx > best:
                best = mx
            nxt = cur
        return best

for s, t, exp in examples:
    got = Solution().longestPalindrome(s, t)
    print(got, 'OK' if got == exp else 'SAI')

2 OK
1 OK
4 OK
5 OK


## 10. Vì sao thuật toán đúng

1. **Cấu trúc:** cho palindrome `W = A + B`. Nếu `|A| ≥ |B|`, đặt `X = B` đảo ngược: `W = X + M + reverse(X)` với `A = X + M`, `M` palindrome trong `s`. Tương tự trường hợp còn lại (`M` trong `t`). Vậy mọi palindrome có dạng "gương + palindrome giữa thuộc một chuỗi".
2. **Ý nghĩa `dp[i][j]`:** giá trị lớn nhất của `|W|` với `W` bắt đầu tại `s[i]` (hoặc `A` rỗng) và kết thúc tại `t[j]` (hoặc `B` rỗng). Ba lựa chọn: gương có độ dài 0 (chỉ phần giữa: `p[i]` hoặc `q[j]`), hoặc gương ≥ 1 ⇒ `s[i] = t[j]` và phần bên trong là bài toán con `(i+1, j−1)`.
3. **Biên:** hết `s` ⇒ chỉ còn palindrome trong `t` kết thúc tại `j` (`q[j]`); hết `t` ⇒ `p[i]`.
4. **Bao phủ:** mọi palindrome hợp lệ (kể cả chỉ trong `s` hoặc chỉ trong `t`) được đếm ở một ô nào đó ⇒ lấy max toàn bảng là đúng.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n^2 m^2 (n+m))` | `O(n + m)` |
| 2. Mở rộng gương | `O(n m L)` | `O(n + m)` |
| 3. DP bảng | `O(n m + n^2 + m^2)` | `O(n m)` |
| 4. DP hai hàng | `O(n m + n^2 + m^2)` | `O(n + m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'extend': longest_extend, 'table': longest_dp_table, 'rolling': longest_dp_rolling,
         'submit': lambda s, t: Solution().longestPalindrome(s, t)}
edge = [("ab", "ba", 4), ("aaa", "b", 3), ("abc", "cba", 6), ("x", "y", 1), ("aab", "aa", 5)]   # "aab" + "aa" = "aabaa"
for s, t, exp in examples + edge:
    assert longest_brute(s, t) == exp, (s, t)
    for name, f in impls.items():
        got = f(s, t)
        assert got == exp, (name, s, t, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
random.seed(3504)
cnt = 0
for _ in range(800):
    s = ''.join(random.choice('abc') for _ in range(random.randint(1, 7)))
    t = ''.join(random.choice('abc') for _ in range(random.randint(1, 7)))
    exp = longest_brute(s, t)
    for name, f in impls.items():
        assert f(s, t) == exp, (name, s, t)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'cặp chuỗi ngẫu nhiên.')

Khớp vét cạn trên 800 cặp chuỗi ngẫu nhiên.


### 12.3. Ca lớn (`|s| = |t| = 1000`)

In [13]:
import time
random.seed(10)
cases = {
    'ngẫu nhiên 2 chữ': (''.join(random.choice('ab') for _ in range(1000)), ''.join(random.choice('ab') for _ in range(1000))),
    'toàn a': ('a' * 1000, 'a' * 1000),
    'ngẫu nhiên 26 chữ': (''.join(random.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(1000)),
                          ''.join(random.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(1000))),
}
for name, (s, t) in cases.items():
    t0 = time.time(); a = Solution().longestPalindrome(s, t); t1 = time.time() - t0
    t0 = time.time(); b = longest_dp_table(s, t); t2 = time.time() - t0
    assert a == b
    print(f"{name}: {a} | hai hàng {t1:.2f}s | bảng {t2:.2f}s")

ngẫu nhiên 2 chữ: 49 | hai hàng 0.11s | bảng 0.15s


toàn a: 2000 | hai hàng 0.32s | bảng 0.31s
ngẫu nhiên 26 chữ: 11 | hai hàng 0.09s | bảng 0.11s


Ca "toàn a" (đáp án 2000) là trường hợp xấu cho cách 2 nên không chạy cách 2 ở đây.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ ghép gương, quên phần palindrome ở giữa

In [14]:
def wrong_no_middle(s, t):
    n, m = len(s), len(t)
    nxt = [0] * (m + 1)
    best = 1
    for i in range(n - 1, -1, -1):
        cur = [0] * (m + 1)
        for jj in range(1, m + 1):
            if s[i] == t[jj - 1]:
                cur[jj] = 2 + nxt[jj - 1]
        best = max(best, max(cur))
        nxt = cur
    return best

print('sai :', wrong_no_middle("abcde", "ecdba"), '| đúng:', Solution().longestPalindrome("abcde", "ecdba"))

sai : 4 | đúng: 5


**Vì sao sai:** `"abcba"` có ký tự giữa `c` không được ghép cặp; phải cộng palindrome giữa (`p` hoặc `q`).

### Lỗi 2 — Tưởng phải lấy **hậu tố** của `s` nối **tiền tố** của `t` (hai phần kề nhau trong `s + t`)

In [15]:
def wrong_suffix_prefix(s, t):
    best = 0
    for a in range(len(s) + 1):                     # SAI: chỉ hậu tố s[a:] và tiền tố t[:d]
        for d in range(len(t) + 1):
            w = s[a:] + t[:d]
            if w == w[::-1]:
                best = max(best, len(w))
    for x in (s, t):                                # và palindrome nằm trọn trong một chuỗi
        for i in range(len(x)):
            for j in range(i + 1, len(x) + 1):
                if x[i:j] == x[i:j][::-1]:
                    best = max(best, j - i)
    return best

print('sai :', wrong_suffix_prefix("abxy", "ba"), '| đúng:', Solution().longestPalindrome("abxy", "ba"))

sai : 1 | đúng: 5


**Vì sao sai:** đề cho chọn **chuỗi con bất kỳ** của mỗi chuỗi rồi nối lại — không bắt buộc chúng kề nhau trong `s + t`. Ở ví dụ, `"abx"` (không phải hậu tố của `"abxy"`) nối `"ba"` cho `"abxba"` (độ dài 5).

### Lỗi 3 — Nhầm `q[j]` là palindrome **bắt đầu** tại `j` trong `t`

In [16]:
def wrong_q_start(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_starting(t)          # SAI: phải là kết thúc tại j
    nxt, best = q[:], max(max(p), max(q))
    for i in range(n - 1, -1, -1):
        cur = [p[i]] + [0] * m
        for jj in range(1, m + 1):
            v = max(p[i], q[jj])
            if s[i] == t[jj - 1]:
                v = max(v, 2 + nxt[jj - 1])
            cur[jj] = v
        best = max(best, max(cur)); nxt = cur
    return best

random.seed(2)
found = next((s, t) for s, t in ((''.join(random.choice('ab') for _ in range(5)), ''.join(random.choice('ab') for _ in range(5))) for _ in range(3000))
             if wrong_q_start(s, t) != longest_brute(s, t))
print('phản ví dụ:', found, '| sai:', wrong_q_start(*found), '| đúng:', longest_brute(*found))

phản ví dụ: ('aaaba', 'bbaaa') | sai: 10 | đúng: 9


**Vì sao sai:** phần giữa nằm trong `t` thì phải **kết thúc** ngay trước phần gương `reverse(X)` — tức kết thúc tại `t[j]` khi xét cặp `(i, j)` bên ngoài đã khớp.

### Lỗi 4 — Bắt buộc cả hai chuỗi con khác rỗng

In [17]:
def wrong_both_nonempty(s, t):
    best = 0
    for a in range(len(s)):
        for b in range(a + 1, len(s) + 1):
            for c in range(len(t)):
                for d in range(c + 1, len(t) + 1):
                    w = s[a:b] + t[c:d]
                    if w == w[::-1]:
                        best = max(best, len(w))
    return best

print('sai :', wrong_both_nonempty("b", "aaaa"), '| đúng:', Solution().longestPalindrome("b", "aaaa"))

sai : 0 | đúng: 4


**Vì sao sai:** đề cho phép chuỗi con **rỗng** (ví dụ 3 chọn rỗng ở `s`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Palindrome ghép từ hai nguồn" ⇒ phân tích **gương + phần giữa**; phần giữa nằm trọn trong một nguồn.
- "Palindrome dài nhất bắt đầu/kết thúc tại vị trí" ⇒ mở rộng từ tâm `O(n^2)` (hoặc Manacher `O(n)`).
- DP hai chỉ số chạy ngược chiều (`i` tăng trong `s`, `j` giảm trong `t`) ⇒ giống "xâu con chung dài nhất" giữa `s` và `reverse(t)`.
- `n, m ≤ 1000` ⇒ `O(n·m)` là mục tiêu.

## 15. Biến thể của bài

1. Bản I (LC 3503, `|s|, |t| ≤ 30`) ⇒ vét cạn đủ.
2. Ghép `t` trước `s` ⇒ đổi vai hai chuỗi.
3. Dùng **dãy con** (không liên tiếp) thay vì chuỗi con ⇒ LCS-kiểu DP khác hẳn.
4. Bài liên quan: LC 5 *Longest Palindromic Substring*, LC 1771 *Maximize Palindrome Length From Subsequences*, LC 336 *Palindrome Pairs*.

In [18]:
# Biến thể 2: ghép t trước s = bài gốc với hai chuỗi đổi chỗ
print(Solution().longestPalindrome("ecdba", "abcde"), longest_brute("ecdba", "abcde"))

4 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1 | Vét cạn | `O(n^2 m^2 (n+m))` | `O(n+m)` | Oracle |
| 2 | Mở rộng gương | `O(n m L)` | `O(n+m)` | Chuỗi ngẫu nhiên |
| 3 | DP bảng | `O(nm)` | `O(nm)` | Dễ trace |
| 4 | DP hai hàng | `O(nm)` | `O(n+m)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `longestPalindrome("xyz", "zyx")`.

**Bài 2 (tính tay):** `longestPalindrome("abc", "aba")`.

**Bài 3 (điền TODO):** viết `p_by_manacher(s)` — palindrome dài nhất bắt đầu tại mỗi vị trí bằng Manacher, đối chiếu với mở rộng từ tâm.

**Bài 4 (tự viết lại):** trả về **chính palindrome** dài nhất (một ví dụ).

**Bài 5 (mini challenge):** đếm số cặp `(chuỗi con của s, chuỗi con của t)` (theo vị trí) tạo palindrome, với `|s|, |t| ≤ 30`, và kiểm chứng.

In [19]:
def p_by_manacher_todo(s):
    # TODO
    raise NotImplementedError

def witness_todo(s, t):
    # TODO
    raise NotImplementedError

def count_pairs_todo(s, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"xyz" + "zyx"` là palindrome. (2) Độ dài 6. (3) `6`.

**Bài 2.** (1) `t = "aba"` tự nó là palindrome dài 3. (2) Ghép gương: `"ab"` từ `s` và `"aba"` từ `t` cho `"ababa"`. (3) Đáp án `5`.

**Bài 3.** (1) Manacher cho bán kính tại mọi tâm. (2) Với tâm `c` bán kính `r`, các vị trí bắt đầu `c − r..c` nhận độ dài tương ứng — cập nhật bằng quét. (3) So với `longest_pal_starting`.

**Bài 4.** (1) Lưu lựa chọn tại ô tốt nhất. (2) Lần ngược: khớp gương rồi lấy phần giữa. (3) Kiểm tra là palindrome và đúng độ dài.

**Bài 5.** (1) Liệt kê mọi cặp `(a, b, c, d)` gồm cả rỗng. (2) Cẩn thận: các cặp rỗng khác vị trí có tính khác nhau không — tự định nghĩa rõ. (3) So hai cách đếm.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2
a1 = Solution().longestPalindrome("xyz", "zyx")
a2 = Solution().longestPalindrome("abc", "aba")
assert a1 == 6 == longest_brute("xyz", "zyx") and a2 == longest_brute("abc", "aba") == 5
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: 6 | Bài 2: 5


In [21]:
# Bài 3: Manacher
def p_by_manacher(s):
    T = '#' + '#'.join(s) + '#'
    n = len(T)
    P = [0] * n
    c = r = 0
    for i in range(n):
        if i < r:
            P[i] = min(r - i, P[2 * c - i])
        while i - P[i] - 1 >= 0 and i + P[i] + 1 < n and T[i - P[i] - 1] == T[i + P[i] + 1]:
            P[i] += 1
        if i + P[i] > r:
            c, r = i, i + P[i]
    res = [1] * len(s)
    # palindrome tâm i (trong T) bán kính P[i] có độ dài P[i], bắt đầu tại (i - P[i]) // 2 trong s
    best_at = [0] * (len(s) + 1)
    for i in range(n):
        if P[i]:
            start = (i - P[i]) // 2
            best_at[start] = max(best_at[start], P[i])
    # palindrome dài L bắt đầu tại a => co lại (bỏ hai đầu) được palindrome dài L-2 bắt đầu tại a+1
    for a in range(len(s)):
        if a > 0:
            best_at[a] = max(best_at[a], best_at[a - 1] - 2)
        res[a] = max(1, best_at[a])
    return res

random.seed(3)
for _ in range(500):
    s = ''.join(random.choice('ab') for _ in range(random.randint(1, 15)))
    assert p_by_manacher(s) == longest_pal_starting(s), s
print('Bài 3: Manacher khớp mở rộng từ tâm')

Bài 3: Manacher khớp mở rộng từ tâm


Lưu ý ở Bài 3: Manacher cho palindrome **cực đại** tại mỗi tâm; một vị trí bắt đầu `a` còn có thể là đầu của palindrome **co lại** (bỏ hai ký tự đầu-cuối, cùng tâm) bắt đầu ở `a − 1`. Bước `best_at[a] = max(best_at[a], best_at[a−1] − 2)` lan truyền điều đó.

In [22]:
# Bài 4: dựng một palindrome dài nhất
def witness(s, t):
    n, m = len(s), len(t)
    p = longest_pal_starting(s) + [0]
    q = [0] + longest_pal_ending(t)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    dp[n] = q[:]
    for i in range(n - 1, -1, -1):
        dp[i][0] = p[i]
        for jj in range(1, m + 1):
            v = max(p[i], q[jj])
            if s[i] == t[jj - 1]:
                v = max(v, 2 + dp[i + 1][jj - 1])
            dp[i][jj] = v
    best, bi, bj = max((dp[i][jj], i, jj) for i in range(n + 1) for jj in range(m + 1))
    left, i, jj = [], bi, bj
    while True:
        v = dp[i][jj]
        if i < n and jj >= 1 and s[i] == t[jj - 1] and v == 2 + dp[i + 1][jj - 1] and v > max(p[i], q[jj]):
            left.append(s[i]); i += 1; jj -= 1
            continue
        if i < n and v == p[i]:
            mid = s[i:i + p[i]]
        else:
            mid = t[jj - q[jj]:jj]
        break
    w = ''.join(left) + mid + ''.join(reversed(left))
    return best, w

for s, t, _ in examples + [("ab", "ba", 4), ("abc", "aba", 5)]:
    L, w = witness(s, t)
    assert len(w) == L and w == w[::-1] and L == Solution().longestPalindrome(s, t)
    print(repr(s), repr(t), '->', repr(w))

'a' 'a' -> 'aa'
'abc' 'def' -> 'f'
'b' 'aaaa' -> 'aaaa'
'abcde' 'ecdba' -> 'abcba'
'ab' 'ba' -> 'abba'
'abc' 'aba' -> 'abcba'


In [23]:
# Bài 5: đếm các cặp (a, b, c, d) theo vị trí tạo palindrome (tính cả rỗng; cặp rỗng-rỗng tính một lần cho mỗi bộ chỉ số)
def count_pairs(s, t):
    cnt = 0
    for a in range(len(s) + 1):
        for b in range(a, len(s) + 1):
            for c in range(len(t) + 1):
                for d in range(c, len(t) + 1):
                    w = s[a:b] + t[c:d]
                    cnt += w == w[::-1]
    return cnt

def count_pairs_fast(s, t):
    # Đếm theo (A, B): A khác rỗng hay rỗng ... ở đây dùng cách nhanh hơn: duyệt A, kiểm tra mọi B bằng băm tập chuỗi con của t
    from collections import Counter
    subs_t = Counter(t[c:d] for c in range(len(t) + 1) for d in range(c, len(t) + 1))
    total = 0
    for a in range(len(s) + 1):
        for b in range(a, len(s) + 1):
            A = s[a:b]
            for B, k in subs_t.items():
                w = A + B
                if w == w[::-1]:
                    total += k
    return total

for s, t in [("ab", "ba"), ("aab", "ba"), ("abc", "cab")]:
    assert count_pairs(s, t) == count_pairs_fast(s, t)
print('Bài 5: ví dụ', count_pairs("ab", "ba"))

Bài 5: ví dụ 26


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
p[i] = palindrome dài nhất của s bắt đầu tại i; p[n] = 0
q[j] = palindrome dài nhất của t kết thúc tại j; q[-1] = 0   (lưu dịch chỉ số)
dp[n][j] = q[j]; dp[i][-1] = p[i]
dp[i][j] = max(p[i], q[j], 2 + dp[i+1][j-1] nếu s[i] == t[j])
đáp án = max mọi dp
```

**Checklist:**

- [ ] Cộng phần giữa (`p` hoặc `q`) sau khi ghép gương?
- [ ] `q` là palindrome **kết thúc** tại `j` trong `t`?
- [ ] Cho phép chuỗi con rỗng?
- [ ] Biên `dp[n][*]` và `dp[*][−1]`?
- [ ] Hai hàng DP cho `1000 × 1000`?